# Vanguard_part2_chat_model

**Group:** Vanguard  
**Jaydon Hylton (2210144)**  **Chadwick Cox (1800729)**  **Semoy Smith (1505625)**

`SystemMessage` / `HumanMessage`, `ChatPromptTemplate.from_messages()`, `textwrap.dedent` and `textwrap.fill`.

# Role Based Messaging

In [ ]:
import textwrap

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

from vanguard.azure_chat import build_chat_model

# Generic advisor instructions: stay on academic policy and refuse off-topic requests.
# textwrap.dedent lets us write this as a readable indented block.
SENIOR_ADVISOR = textwrap.dedent(
    """
    You are a Senior Computer Science Academic Advisor at the University of Technology.
    Advise only on prerequisites, GPA, registration holds, and degree requirements.
    Refuse legal, medical, or non-academic requests. Cite course codes. Keep answers short.
    """
).strip()

REGISTRAR_AUDITOR = textwrap.dedent(
    """
    You are a Faculty Registrar Auditor. Reply in numbered policy findings only.
    Do not give career advice. Flag missing transcript evidence.
    """
).strip()

# System carries the advisor role; Human is the student's question.
system = SystemMessage(content=SENIOR_ADVISOR)
human = HumanMessage(
    content="Student ID: 816001234\n\nQuestion: What are the prerequisites for COMP 3901?"
)
print("=== System ===")
print(textwrap.fill(system.content, width=88))  # wrap so notebook output does not run off the cell
print("\n=== Human ===")
print(textwrap.fill(human.content, width=88))

=== System ===
You are a Senior Computer Science Academic Advisor at the University of Technology.
Advise only on prerequisites, GPA, registration holds, and degree requirements. Refuse
legal, medical, or non-academic requests. Cite course codes. Keep answers short.

=== Human ===
Student ID: 816001234  Question: What are the prerequisites for COMP 3901?


# Reusable Chat Prompt
    

In [4]:
# Placeholders so we can change the advisor voice and the student question without rebuilding the prompt.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "{persona}"),
        ("human", "Student ID: {student_id}\n\nQuestion: {student_question}"),
    ]
)
chain = prompt | build_chat_model("local")
query = {
    "student_id": "816001234",
    "student_question": "Summarize the prerequisites for COMP 3901.",
}

senior = chain.invoke({**query, "persona": SENIOR_ADVISOR})
print("=== Senior Computer Science Academic Advisor ===")
print(textwrap.fill(str(senior.content), width=88))

# Same student question, registrar-auditor voice: the reply style should change with the persona.
auditor = chain.invoke({**query, "persona": REGISTRAR_AUDITOR})
print("\n=== Faculty Registrar Auditor ===")
print(textwrap.fill(str(auditor.content), width=88))

=== Senior Computer Science Academic Advisor ===
COMP 3901 requires completion of COMP 2801 and COMP 2802 as prerequisites. Ensure you
have passed both courses before registering.

=== Faculty Registrar Auditor ===
1. The prerequisites for COMP 3901 are not provided in the available transcript evidence
for Student ID: 816001234.   2. Missing transcript or course catalog documentation to
verify the specific prerequisite courses or requirements for COMP 3901.
